# 08 — Experiment 4c DPO

**Variable under test:** DPO preference learning on top of exp4c-sft.

**Config:** DPOTrainer, r=16, alpha=32, beta=0.1, 3 epochs, LR 5e-6, batch 1 × grad_accum 4 (effective 4), max_length=2048, max_prompt_length=1024, 4-bit base, vision layers unfrozen, target_modules=all-linear.

**Base SFT:** exp4c-sft (`/mnt/civicinsight/checkpoints/exp4c-sft/checkpoint-80`) — NOT exp4b. exp4c-sft contains the choropleth augmentation fix; training DPO on exp4b would silently revert that.

**Inputs:** ~50-150 preference pairs from `dataset.marked.json` (61 golds × 10 perturbations, filtered to non-no-op + over-sampling on critical failure modes — `proper_noun_prior_substitution`, `fabricate_external_fact`, `force_clean_structure`, `positional_schema_swap`).

**What to watch for:** targeted improvements on the audit failure classes (positional binding on rural-vs-urban, fabricated external facts on income-vs-life-exp, false_selection on browser-share, proper-noun substitution Sartène→Ajaccio class). No regression on chpth-1's exp4c-sft fix. Internal consistency (no "tooltip visible" / "no tooltip" contradictions in same output).

**If/Then** → If DPO loss stays near ln(2) (~0.69), preference signal isn't strong enough. Investigate beta. → If targeted improvements show on at least one of (rural-vs-urban, income-vs-life-exp) AND no chpth-1 regression, ship as exp4c-dpo. → Otherwise exp4c-sft remains canonical.

In [1]:
!mkdir -p /mnt/civicinsight/standardized /mnt/civicinsight/training /mnt/civicinsight/test /mnt/civicinsight/checkpoints/exp4c-dpo /mnt/civicinsight/results
!ls -al /mnt/civicinsight/

total 6128
drwxr-xr-x 1 root root     134 Apr 30 05:32 .
drwxr-xr-x 3 root root      47 May  1 17:37 ..
drwxr-xr-x 1 root root      37 Apr 29 05:11 checkpoints
drwxr-xr-x 1 root root       0 Apr 26 06:33 hf_cache
drwxr-xr-x 1 root root     145 Apr 23 13:23 model
-rw-r--r-- 1 root root 6259526 Apr 26 21:32 raw_archive.tar.gz
-rw-r--r-- 1 root root    4416 May  1 16:48 requirements-dpo-v1-WORKING.txt
-rw-r--r-- 1 root root    4341 Apr 29 05:51 requirements-exp4c-WORKING.txt
drwxr-xr-x 1 root root      58 Apr 30 05:42 results
drwxr-xr-x 1 root root    1914 Apr 29 04:19 standardized
drwxr-xr-x 1 root root     134 Apr 29 04:19 test
drwxr-xr-x 1 root root      19 Apr 29 04:14 training


In [2]:
# Same install pattern as dpo-vision-repro, but with strict pins on top-level
# deps to defend against version drift since verification.
%uv pip install pillow==11.3.0 --system
%uv pip install transformers==5.5.0 --system
%uv pip install trl==0.24.0 --system
%uv pip install peft==0.19.1 --system
%uv pip install datasets==4.3.0 --system
%uv pip install unsloth-zoo==2026.4.9 --system

# Pin to official Unsloth mainline post-merge of PR #5199 (Apr 28).
# Earlier pin to datta0/unsloth@e96d05ba was orphaned by force-push during PR
# cleanup — commit ID still resolves but is no longer reachable from any branch
# and will eventually be GC'd. Mainline merge commit is immutable.
# This pin includes all three vision DPO fixes from issue #5196:
#   - tokenization hang (a9729c8)
#   - data collator schema (6b11713)
#   - vision keys handling (dc3e6a5 + dpo_trainer_data_collator_vision_keys)
%uv pip uninstall unsloth --system
%uv pip install "git+https://github.com/unslothai/unsloth.git@4f9c8321a2136e62fd86fe722a544afd534334a5" --no-deps --system

# Force torch+torchvision matching pair from cu129 index, last
%uv pip install --reinstall torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu129 --system

!rm -rf /root/unsloth_compiled_cache

%uv pip install bitsandbytes==0.49.2 --system


# Verify mainline pin is actually installed at the expected commit.
import glob, json
EXPECTED_COMMIT = "4f9c8321a2136e62fd86fe722a544afd534334a5"
verified = False
for path in glob.glob("/usr/local/lib/python3.12/site-packages/unsloth-*.dist-info/direct_url.json"):
    with open(path) as f:
        info = json.load(f)
    print(f"=== {path} ===")
    print(json.dumps(info, indent=2))
    url = info.get("url", "")
    commit = info.get("vcs_info", {}).get("commit_id", "")
    if "unslothai/unsloth" in url and commit == EXPECTED_COMMIT:
        verified = True

if verified:
    print(f"\nMainline pin {EXPECTED_COMMIT[:8]} verified. RESTART KERNEL before importing unsloth.")
else:
    print(f"\nMainline pin NOT verified at {EXPECTED_COMMIT[:8]}. Install may have failed silently — check output above.")

Using Python 3.12.6 environment at: /usr/local
Resolved 1 package in 103ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
Prepared 1 package in 194ms
Uninstalled 1 package in 11ms
Installed 1 package in 101ms
 - pillow==11.0.0
 + pillow==11.3.0
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 27 packages in 310ms
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)

In [3]:
# imports — mirrors exp4 pattern, plus trl DPO bits and peft helpers for manual SFT weight loading

from unsloth import FastVisionModel          # MUST be first
from trl import DPOTrainer, DPOConfig
from peft import set_peft_model_state_dict
from safetensors.torch import load_file as safe_load
import torch
import os
import json
import re
import random
from PIL import Image
from huggingface_hub import login, HfApi, create_repo
import time

# from 01-zeroshot tests
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

random.seed(42)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in via Modal Secret")
else:
    print("HF_TOKEN not found in environment. Add it as a Modal Secret.")


Logged in via Modal Secret


In [5]:
SFT_CHECKPOINT = "/mnt/civicinsight/checkpoints/exp4c-sft/checkpoint-80"

# Load 4-bit base — exact same call as exp4 cell 6
model, tokenizer = FastVisionModel.from_pretrained(
    "/mnt/civicinsight/model",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print(f"Base loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Attach a FRESH LoRA adapter using exp4's exact config.
# We do this instead of PeftModel.from_pretrained because the latter rejects
# Gemma4ClippableLinear at module-resolution time (it lands on the wrapper
# instead of descending to the inner .linear). FastVisionModel.get_peft_model
# goes through Unsloth's wrapper which handles the descent correctly. Then we
# copy the SFT-trained weights into the freshly-attached adapter.
model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    target_modules="all-linear",
)
print(f"Fresh adapter attached. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("Trainable params (fresh, before SFT load):")
model.print_trainable_parameters()

# Now copy SFT-trained LoRA weights into the freshly-attached adapter.
# Adapter shapes match because we used identical r/alpha/target_modules to exp4.
sft_weights_path = os.path.join(SFT_CHECKPOINT, "adapter_model.safetensors")
sft_state = safe_load(sft_weights_path)
print(f"\nLoaded {len(sft_state)} tensors from {sft_weights_path}")

load_result = set_peft_model_state_dict(model, sft_state)

print(f"\nGPU after SFT weight load: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("Trainable params (after SFT load — should be unchanged from above):")
model.print_trainable_parameters()

# Freeze vision LoRA so DPO only updates language-side params.
# SFT-trained vision weights stay loaded — they just stop receiving gradients.
vision_frozen_params = 0
language_trainable_params = 0
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if "vision_tower" in name:
        param.requires_grad = False
        vision_frozen_params += param.numel()
    else:
        language_trainable_params += param.numel()

print(f"\nVision LoRA frozen: {vision_frozen_params:,} params")
print(f"Language LoRA trainable: {language_trainable_params:,} params")
print("\nFinal trainable params (after vision freeze):")
model.print_trainable_parameters()


==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu129. CUDA: 8.0. CUDA Toolkit: 12.9. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

/usr/local/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Base loaded. GPU: 10.0 GB
Fresh adapter attached. GPU: 10.2 GB
Trainable params (fresh, before SFT load):
trainable params: 41,222,144 || all params: 8,037,378,592 || trainable%: 0.5129

Loaded 740 tensors from /mnt/civicinsight/checkpoints/exp4c-sft/checkpoint-80/adapter_model.safetensors

GPU after SFT weight load: 10.2 GB
Trainable params (after SFT load — should be unchanged from above):
trainable params: 41,222,144 || all params: 8,037,378,592 || trainable%: 0.5129

Vision LoRA frozen: 4,521,984 params
Language LoRA trainable: 36,700,160 params

Final trainable params (after vision freeze):
trainable params: 36,700,160 || all params: 8,037,378,592 || trainable%: 0.4566


In [6]:
# Verify SFT weights actually got copied (not just counted as "loaded")
# Pick one LoRA weight tensor and confirm it's not zero-init
for name, param in model.named_parameters():
    if "lora_A" in name and "vision_tower.encoder.layers.0" in name:
        print(f"{name}")
        print(f"  shape: {tuple(param.shape)}")
        print(f"  abs mean: {param.abs().mean().item():.6f}")
        print(f"  abs max:  {param.abs().max().item():.6f}")
        break

base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.linear.lora_A.default.weight
  shape: (16, 768)
  abs mean: 0.018263
  abs max:  0.039144


In [7]:
# Pre-DPO sanity check: vision frozen, language hot, SFT vision weights intact.
n_vision_trainable = sum(p.numel() for n, p in model.named_parameters()
                         if p.requires_grad and "vision_tower" in n)
n_language_trainable = sum(p.numel() for n, p in model.named_parameters()
                           if p.requires_grad and "vision_tower" not in n)

assert n_vision_trainable == 0, f"Vision still has {n_vision_trainable:,} trainable params"
assert n_language_trainable > 0, "No language params trainable — SFT adapter didn't load?"

# Confirm SFT-trained vision LoRA is non-zero (would indicate the load wiped it).
sample_vision = next(
    p for n, p in model.named_parameters()
    if "vision_tower" in n and "lora_A" in n
)
assert sample_vision.abs().mean().item() > 0.001, (
    f"Vision LoRA mean abs={sample_vision.abs().mean().item():.6f} — "
    "looks zero-init, SFT vision weights may have been wiped by freeze."
)

print(f"Vision trainable params:   {n_vision_trainable:,}  (must be 0)")
print(f"Language trainable params: {n_language_trainable:,}")
print(f"Sample vision LoRA |mean|: {sample_vision.abs().mean().item():.6f}  (must be > 0.001)")
print("\nReady for DPO. Vision frozen, language hot, SFT vision intact.")


Vision trainable params:   0  (must be 0)
Language trainable params: 36,700,160
Sample vision LoRA |mean|: 0.018263  (must be > 0.001)

Ready for DPO. Vision frozen, language hot, SFT vision intact.


In [8]:
import random
import re

# perturbation library. a collection of helper functions to
# manipulate our gold text for DPO
MARKER = "[civicinsight-v1]"
BANNED_HEDGES = ["around ", "approximately ", "roughly ", "about "]
TOOLTIP_TEXT = "tooltip"
NO_TOOLTIP_TEXT = ". No tooltip is visible."
OPENER_SLOT = "This "

def strip_marker(gold: str) -> str:             # returns the gold string with the MARKER removed
    return gold.replace(f"{MARKER} ", "")

def inject_hedge(gold: str) -> str:
    # find number in string
    num = re.search(r"\b\d+\b", gold)
    if not num:
        return gold
    hedge = random.choice(BANNED_HEDGES)
    return gold[:num.start()] + hedge + gold[num.start():]

def break_consistency(gold: str) -> str:
    if TOOLTIP_TEXT not in gold.lower():
        if "selected" not in gold.lower():
            return gold.rstrip(".") + ". An element is selected."
        return gold
    variations = [
        ". No tooltip is visible.",
        ". Tooltips are closed.",
        ". Nothing is selected."
    ]
    return gold.rstrip(".") + random.choice(variations)

def wrong_slot(gold: str) -> str:
    if OPENER_SLOT not in gold:
        return gold
    return gold.replace(OPENER_SLOT, random.choice(["A ", ""]), 1)

def positional_schema_swap(gold: str) -> str:
    matches = list(re.finditer(r"\b(\d{1,3})%", gold))
    if len(matches) < 2:
        return gold

    m1, m2 = matches[0], matches[-1]
    if m1.start(1) == m2.start(1):
        return gold
    v1 = m1.group(1)
    v2 = m2.group(1)
    return (
        gold[:m1.start(1)]
        + v2
        + gold[m1.end(1):m2.start(1)]
        + v1
        + gold[m2.end(1):]
    )

def googly_wrap(gold: str) -> str:
    body = gold.replace(MARKER, "").strip()
    return f"**Aria Label:**\n\n* {body}\n\nThis description is provided for screen readers."

def proper_noun_prior_substitution(gold: str) -> str:
    subs = {
        "Sartène": "Ajaccio",
        "Revin": "Charleville-Mézières",
        "Ploërmel": "Rennes",
        "Plœrières": "Rennes",
        "Cléguérec": "Vannes",
        "Rostrenen": "Brest",
        "Gourin": "Lorient",
        "Mérignac": "Bordeaux",
        "Tourcoing": "Lille",
        "Aix-en-Provence": "Marseille",
        "Saint-Germain-en-Laye": "Versailles",
        "Vénissieux": "Lyon",
        "Bastia": "Ajaccio"
    }
    for k, v in subs.items():
        if k in gold:
            return gold.replace(k, v)
    return gold

def fabricate_external_fact(gold: str) -> str:
    facts = {
        "Ireland": "105 000 US-Dollars",
        "United States": "85 000 US-Dollars"
    }
    for country, fake_value in facts.items():
        if country in gold:
            return re.sub(r"(\d{1,3}(?: \d{3})*) US-Dollars", fake_value, gold)
    return gold

def force_clean_structure(gold: str) -> str:
    if "selected with the tooltip" in gold:
        return re.sub(r"A .*? is selected with the tooltip: .*?\.", "Tooltips are visible for all categories showing exact values.", gold)
    match = re.search(r"(\d+|Two|Three|Four|Five|Six)\s+tooltips?\s+(?:are\s+)?visible", gold, re.IGNORECASE)
    if match:
        return re.sub(r"(\d+|Two|Three|Four|Five|Six)\s+tooltips?\s+(?:are\s+)?visible[^.]*\.", "Tooltips are visible for all categories with exact values.", gold, flags=re.IGNORECASE)
    return gold

def color_label_substitution(gold: str) -> str:
    COLORS = ["blue", "black", "red", "green", "yellow", "orange", "purple", "teal", "gray"]
    found_colors = [c for c in COLORS if c in gold.lower()]
    if len(found_colors) >= 2:
        c1, c2 = found_colors[0], found_colors[1]
        temp = gold.replace(c1, "XYZ_TEMP_C1").replace(c1.capitalize(), "XYZ_TEMP_C1_CAP")
        temp = temp.replace(c2, c1).replace(c2.capitalize(), c1.capitalize())
        temp = temp.replace("XYZ_TEMP_C1_CAP", c2.capitalize()).replace("XYZ_TEMP_C1", c2)
        return temp
    return gold

PERTURBATIONS = {
    "strip_marker": strip_marker,
    "wrong_slot": wrong_slot,
    "inject_hedge": inject_hedge,
    "positional_schema_swap": positional_schema_swap,
    "break_consistency": break_consistency,
    "googly_wrap": googly_wrap,
    "proper_noun_prior_substitution": proper_noun_prior_substitution,
    "fabricate_external_fact": fabricate_external_fact,
    "force_clean_structure": force_clean_structure,
    "color_label_substitution": color_label_substitution,
}

# Sanity check — preview each perturbation against representative golds.
# Reads first few entries of dataset.marked.json so we can see real before/after,
# and flags any perturbation that no-ops on the sample (potentially under-targeted).
import json as _json
with open("/mnt/civicinsight/training/dataset.marked.json") as _f:
    _samples = _json.load(_f)

print("=== Perturbation sanity check (showing first triggering sample per type) ===\n")
for _name, _fn in PERTURBATIONS.items():
    _triggered = False
    for _s in _samples:
        _gold = _s["aria_label"]
        _rejected = _fn(_gold)
        if _rejected != _gold:
            _triggered = True
            print(f"--- {_name} ---")
            print(f"  GOLD:     {_gold[:120]}...")
            print(f"  REJECTED: {_rejected[:120]}...")
            print()
            break
    if not _triggered:
        print(f"--- {_name} ---")
        print(f"  (no-op on all {len(_samples)} samples — perturbation under-targeted or specific to held-outs)")
        print()


=== Perturbation sanity check (showing first triggering sample per type) ===

--- strip_marker ---
  GOLD:     [civicinsight-v1] This line chart titled 'Émissions mondiales de CO2 par type de carburant et par secteur d'activité' sh...
  REJECTED: This line chart titled 'Émissions mondiales de CO2 par type de carburant et par secteur d'activité' shows global CO2 emi...

--- wrong_slot ---
  GOLD:     [civicinsight-v1] This line chart titled 'Émissions mondiales de CO2 par type de carburant et par secteur d'activité' sh...
  REJECTED: [civicinsight-v1] A line chart titled 'Émissions mondiales de CO2 par type de carburant et par secteur d'activité' shows...

--- inject_hedge ---
  GOLD:     [civicinsight-v1] This line chart titled 'Émissions mondiales de CO2 par type de carburant et par secteur d'activité' sh...
  REJECTED: [civicinsight-v1] This line chart titled 'Émissions mondiales de CO2 par type de carburant et par secteur d'activité' sh...

--- positional_schema_swap ---
  GOLD:    

In [9]:

PROMPT = "Generate an aria-label for this data visualization image."
# DPO uses raw-string prompts (no apply_chat_template), so the image-token
# placeholder must be inlined here. Without it, input_ids has 0 image-token
# slots while the vision encoder produces 512 features, and the first DPO
# step crashes with: "Image features and image tokens do not match,
# tokens: 0, features: 512". Cells 9 and 14 keep using PROMPT (clean) because
# apply_chat_template inserts the image token from the conversation structure.
DPO_PROMPT = f"{tokenizer.image_token}\n{PROMPT}"
DATASET_JSON = "/mnt/civicinsight/training/dataset.marked.json"
STD_TRAIN_IMAGE_DIR = "/mnt/civicinsight/standardized"
ROTATION = ["strip_marker", "wrong_slot", "inject_hedge", "break_consistency", "positional_schema_swap", "googly_wrap", "proper_noun_prior_substitution", "fabricate_external_fact", "force_clean_structure", "color_label_substitution"]

def make_pair(image, gold, rejected, prompt=DPO_PROMPT):
    return {
        "images": [image],
        "prompt": prompt,
        "chosen": gold,
        "rejected": rejected,
    }

pairs = []
with open(DATASET_JSON) as f:
    dataset = json.load(f)

from collections import Counter
counter = Counter()

for i, record in enumerate(dataset):
    img = Image.open(os.path.join(STD_TRAIN_IMAGE_DIR, os.path.basename(record["image"])))
    gold = record["aria_label"]
    pert_name = ROTATION[i % len(ROTATION)]
    rejected = PERTURBATIONS[pert_name](gold)

    if rejected != gold:
        pairs.append(make_pair(img, gold, rejected))
        counter[pert_name] += 1

# Over-represent positional_schema_swap
for record in dataset:
    img = Image.open(os.path.join(STD_TRAIN_IMAGE_DIR, os.path.basename(record["image"])))
    gold = record["aria_label"]
    rejected = positional_schema_swap(gold)
    if rejected != gold:
        pairs.append(make_pair(img, gold, rejected))
        counter["positional_schema_swap"] += 1

# Over-represent critical perturbations
for record in dataset:
    img = Image.open(os.path.join(STD_TRAIN_IMAGE_DIR, os.path.basename(record["image"])))
    gold = record["aria_label"]
    
    for pert in ["proper_noun_prior_substitution", "fabricate_external_fact", "force_clean_structure"]:
        rejected = PERTURBATIONS[pert](gold)
        if rejected != gold:
            pairs.append(make_pair(img, gold, rejected))
            counter[pert] += 1

print(f"DPO_PROMPT: {DPO_PROMPT!r}")
print(f"Total pairs: {len(pairs)}")
print(f"\nPerturbation distribution:")
for name, n in counter.most_common():
    print(f"  {name:<35} {n}")


DPO_PROMPT: '<|image|>\nGenerate an aria-label for this data visualization image.'
Total pairs: 56

Perturbation distribution:
  proper_noun_prior_substitution      12
  positional_schema_swap              11
  strip_marker                        7
  wrong_slot                          6
  inject_hedge                        6
  googly_wrap                         6
  break_consistency                   5
  color_label_substitution            2
  force_clean_structure               1


In [10]:
# ============================================================
# Sweep the Held-out — run pre-DPO inference on EVERY image in the test dir
# ============================================================
# switch model to eval
model.eval()

STD_TEST_IMG_DIR = "/mnt/civicinsight/test"
prompt = "Generate an aria-label for this data visualization image."

test_images = sorted([f for f in os.listdir(STD_TEST_IMG_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))])
print(f"Found {len(test_images)} held-out images\n")

before_dpo_results = {}


for img_name in test_images:
    img_path = os.path.join(STD_TEST_IMG_DIR, img_name)
    img = Image.open(img_path)
    message = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=600, do_sample=False)
    elapsed = time.time() - start
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    before_dpo_results[img_name] = decoded
    print(f"{'='*70}")
    print(f"IMAGE: {img_name} | {elapsed:.1f}s")
    print(f"{'='*70}")
    print(decoded)
    print()

# eval is done, back to training mode
model.train()

Found 7 held-out images



/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bits

IMAGE: 04-dept-yvelines.png | 130.6s
user
Generate an aria-label for this data visualization image.
model
[civicinsight-v1] This choropleth map titled 'Yvelines — communes ayant atteint le 2e tour' shows communes colored by the winning political bloc. The legend on the bottom right shows: Bloc vainqueur - Dark Red square for Extrême gauche, Pink square for Gauche, Yellow square for Centre, Gray square for Divers, Blue square for Droite and Black square for Extrême droite. 18 communes are shown colored: Gauche (Pink) - 3, Centre (Yellow) - 2, Divers (Gray) - 5, Droite (Blue) - 6 and Extrême droite (Black) - 2.

IMAGE: baseline-1.png | 71.9s
user
Generate an aria-label for this data visualization image.
model
[civicinsight-v1] This choropleth map titled 'Ploeren' shows arrondissements colored by the real estate price per square metre. The legend on the right shows color scale from 0 - 1076 EUR/m2 (light pink) to 2989 - 6571 EUR/m2 (dark red). 12 arrondissements are shown colored: dark re

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (language_model): Gemma4TextModel(
          (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
          (layers): ModuleList(
            (0-4): 5 x Gemma4TextDecoderLayer(
              (self_attn): Gemma4TextAttention(
                (q_norm): Gemma4RMSNorm()
                (k_norm): Gemma4RMSNorm()
                (v_norm): Gemma4RMSNorm()
                (k_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=2560, out_features=512, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=2560, out_features=16, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=16, out_features=512

In [11]:
import os
os.makedirs("/mnt/civicinsight/hf_cache", exist_ok=True)
os.environ["HF_DATASETS_CACHE"] = "/mnt/civicinsight/hf_cache"
print(f"Cache set to: {os.environ['HF_DATASETS_CACHE']}")
print(f"  exists: {os.path.exists(os.environ['HF_DATASETS_CACHE'])}")
print(f"  writable: {os.access(os.environ['HF_DATASETS_CACHE'], os.W_OK)}")

Cache set to: /mnt/civicinsight/hf_cache
  exists: True
  writable: True


In [12]:
import time
from datasets import Dataset

# Let datasets auto-detect types from PIL objects in `pairs`. Earlier
# attempt at explicit Features({"images": [HFImage()], ...}) caused the
# DPOTrainer to error with mixed keys (images + pixel_values both present)
# because HFImage byte-encoding doesn't match what the vision DPO pipeline
# expects. Working dpo-vision-repro uses no explicit features here.
start = time.time()
pairs_ds = Dataset.from_list(pairs)
print(f"Built dataset in {time.time()-start:.1f}s")
print(f"Columns: {pairs_ds.column_names}")
print(f"First image: {type(pairs_ds[0]['images'][0])}")


Built dataset in 0.0s
Columns: ['images', 'prompt', 'chosen', 'rejected']
First image: <class 'PIL.PngImagePlugin.PngImageFile'>


In [14]:
# Pre-flight: confirm prompts have image-token slots before training.
# If this asserts to 0, DPOTrainer.train() will fail at first step with:
#   ValueError: Image features and image tokens do not match, tokens: 0, features: 512
# Catching it here saves the 15-minute dry run.
sample = pairs_ds[0]
processed = tokenizer(
    text=sample["prompt"] + sample["chosen"],
    images=sample["images"],
    return_tensors="pt",
)
n_image_tokens = (processed["input_ids"] == tokenizer.image_token_id).sum().item()
assert n_image_tokens > 0, (
    f"No image tokens in input_ids — prompt missing {tokenizer.image_token!r}. "
    f"Check that DPO_PROMPT in cell 8 prepends tokenizer.image_token."
)
print(f"OK: {n_image_tokens} image-token slots in input_ids, "
      f"pixel_values shape {tuple(processed['pixel_values'].shape)}")

OK: 256 image-token slots in input_ids, pixel_values shape (1, 2520, 768)


In [15]:
# DPO training run — main event
# Replaces SFTTrainer with DPOTrainer; uses preference pairs from cell 7


dpo_trainer = DPOTrainer(
    model=model,                                    # policy (will be trained)
    ref_model=None,                                 # ref_model=None → DPOTrainer auto-creates frozen reference from current model state
    processing_class=tokenizer,                     # if newer TRL errors with TypeError: swap to processing_class=tokenizer
    train_dataset=pairs_ds,                         # preference pairs from cell 7, converted to Dataset
    data_collator=None,
    args=DPOConfig(
        beta=0.1,                                   # KL penalty strength (DPO-specific; standard starting point)
        per_device_train_batch_size=1,              # keep simple for v1; A100 has headroom for more in v2
        gradient_accumulation_steps=4,              # effective batch of 4
        num_train_epochs=3,                         # DPO converges faster than SFT (1-3 epochs typical)
        learning_rate=5e-6,                         # ~40x lower than SFT's 2e-4 (standard for DPO)
        output_dir="/mnt/civicinsight/checkpoints/exp4c-dpo",
        max_length=2048,                            # max tokens per example (image + text combined)
        max_prompt_length=1024,                     # DPO-specific: max tokens for the prompt portion
        remove_unused_columns=False,                # required for image+text format
        logging_steps=1,                            # log every step (small dataset, want fine-grained loss curve)
        dataset_num_proc=1,                         # Modal reports os.cpu_count()=47 but allocates ~2 cores;
                                                    # default explodes RAM (40GB+) and stalls. Keep small.
    ),
)

print(f"GPU before DPO: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"Total preference pairs: {len(pairs)}")
print(f"\nStarting DPO training...")
dpo_trainer.train()
print(f"\nDPO training ran without crash!")
print(f"GPU after DPO: {torch.cuda.memory_allocated()/1e9:.1f} GB")


Extracting prompt in train dataset (num_proc=1):   0%|          | 0/56 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=1):   0%|          | 0/56 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/56 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


GPU before DPO: 10.2 GB
Total preference pairs: 56

Starting DPO training...


...

Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/site-packages/bits

Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
1,0.381214,-21.267467,-24.309746,0.750000,3.042279,-2471.297363,-2479.481201,-7.870996,-7.760978
2,1.307350,-20.170429,-20.256042,0.500000,0.085613,-3848.040771,-3175.884277,-8.108274,-8.087011
3,0.292147,-20.398157,-22.345068,0.750000,1.946912,-2881.376221,-2914.989746,-8.406183,-8.436044
4,0.469234,-15.264200,-15.851501,0.750000,0.587302,-2354.626465,-2376.314941,-8.398541,-8.432571
5,3.625712,-19.150826,-15.720829,0.000000,-3.429996,-2583.592773,-2501.688232,-8.351190,-8.113264
6,0.227772,-18.703449,-20.741138,1.000000,2.037690,-2718.354248,-2754.467773,-8.243549,-8.003987
7,0.516724,-15.931403,-18.417427,0.750000,2.486023,-3395.176758,-3384.970215,-7.900982,-7.934680
8,2.015069,-23.650433,-22.785946,0.250000,-0.864489,-3314.163086,-3285.255859,-7.835871,-7.791681
9,0.313616,-17.689398,-19.042692,0.750000,1.353293,-3542.769287,-3565.979248,-7.914032,-7.904816
10,0.334438,-20.089031,-24.731461,0.750000,4.642429,-3640.693359,-3648.954346,-8.513232,-8.461596


Unsloth: Restored added_tokens_decoder metadata in /mnt/civicinsight/checkpoints/exp4c-dpo/checkpoint-42/tokenizer_config.json.



DPO training ran without crash!
GPU after DPO: 10.3 GB


In [16]:
# Snapshot the working DPO stack post-training. Separate file from the SFT
# requirements because DPO uses datta0 unsloth fork + trl 0.24.0 + transformers
# 5.5.0, all of which differ from the SFT stack.
import subprocess, os

result = subprocess.run(
    ["pip", "freeze"],
    capture_output=True, text=True,
    env={**os.environ, "NO_COLOR": "1", "PIP_NO_COLOR": "1"},
)
out_path = "/mnt/civicinsight/requirements-dpo-v1-WORKING.txt"
with open(out_path, "w") as f:
    f.write(result.stdout)
print(f"Saved {len(result.stdout.splitlines())} pinned packages to {out_path}")


Saved 236 pinned packages to /mnt/civicinsight/requirements-dpo-v1-WORKING.txt


In [17]:
# AFTER-DPO held-out sweep — same 5 images, same prompt, same do_sample=False as cell 8
# Direct comparison anchor against before_dpo_results
model.eval()

after_dpo_results = {}
for img_name in test_images:
    img_path = os.path.join(STD_TEST_IMG_DIR, img_name)
    img = Image.open(img_path)
    message = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": PROMPT},
    ]}]
    inputs = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=600, do_sample=False)
    elapsed = time.time() - start
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    after_dpo_results[img_name] = decoded
    print(f"{'='*70}")
    print(f"AFTER-DPO | {img_name} | {elapsed:.1f}s")
    print(f"{'='*70}")
    print(decoded)
    print()

/usr/local/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


AFTER-DPO | 04-dept-yvelines.png | 49.9s
user
Generate an aria-label for this data visualization image.
model
[civicinsight-v1] This choropleth map titled 'Yvelines — communes ayant atteint le 2e tour' shows communes colored by the winning political bloc. The legend on the bottom right shows: Bloc vainqueur - Dark Red square for Extrême gauche, Pink square for Gauche, Yellow square for Centre, Gray square for Divers, Blue square for Droite and Black square for Extrême droite. 18 communes are shown colored: Gauche (Pink) - 3, Centre (Yellow) - 2, Divers (Gray) - 5, Droite (Blue) - 6 and Extrême droite (Black) - 2.

AFTER-DPO | baseline-1.png | 43.4s
user
Generate an aria-label for this data visualization image.
model
[civicinsight-v1] This choropleth map titled 'Ploeren' shows arrondissements colored by the real estate price per square metre. The legend on the right shows color scale from 0 - 1076 EUR/m2 (light pink) to 2989 - 6571 EUR/m2 (dark red). 12 arrondissements are shown colored

In [18]:
# Extended scorecard — existing 3 metrics + 2 NEW (positional-schema, consistency)
# Compares before-DPO vs after-DPO across all 5 held-outs

# Existing (copied from 07-experiment-4 cell 12)
SLOT_OPENERS = ("This line", "This bar", "This stacked", "This box", "This scatter",
                "This choropleth", "This hexagonal", "This panel", "This small multiples",
                "This untitled", "This area", "This pie", "This gauge", "This table")
BANNED_ADJECTIVES = ("around ", "approximately", "roughly ", "about ", "appears to",
                     "notably", "dramatically", "rises steadily", "drops steadily",
                     "significantly", "suggesting")

# NEW: positional-schema known-wrong segment-value bindings per held-out image
# format: list of (label_substr, wrong_value_substr) — if BOTH appear within 100 chars, flag
POSITIONAL_KNOWN_WRONGS = {
    "rural-vs-urban.png": [
        ("China", "53%"),      # China capital ~7-10%; the 53% is on other-urban
        ("USA", "80%"),        # USA capital ~10%; the 80% is actually on the other-urban segment
        ("Australia", "88%"),  # Australia capital ~8%; the 88% is on other-urban
        ("India", "30%"),      # India capital ~2%; the 30% is on the other-urban segment
    ],
}

# NEW: internal-consistency contradiction patterns (regex pairs that should NOT both appear in same output)
CONSISTENCY_CONTRADICTIONS = [
    (r"tooltip\s+is\s+visible", r"no\s+tooltip\s+is\s+visible"),
    (r"is\s+selected", r"no\s+\w+\s+is\s+selected"),
]

def score(output, img_name=None):
    assistant_part = output.split("model\n")[-1] if "model\n" in output else output
    assistant_part = assistant_part.strip()
    lower = assistant_part.lower()

    has_marker = MARKER in assistant_part
    opens_with_slot = any(
        assistant_part.lstrip().startswith(MARKER + " " + s)
        or assistant_part.lstrip().startswith(s)
        for s in SLOT_OPENERS
    )
    banned_found = [a.strip() for a in BANNED_ADJECTIVES if a in lower]

    # NEW: positional-schema (only flagged for held-outs in POSITIONAL_KNOWN_WRONGS)
    positional_errors = []
    if img_name and img_name in POSITIONAL_KNOWN_WRONGS:
        for s1, s2 in POSITIONAL_KNOWN_WRONGS[img_name]:
            if s1 in assistant_part and s2 in assistant_part:
                idx1 = assistant_part.index(s1)
                idx2 = assistant_part.index(s2)
                if abs(idx1 - idx2) < 100:  # within 100 chars = likely paired
                    positional_errors.append(f"{s1}+{s2}")

    # NEW: internal consistency
    consistency_violations = []
    for p1, p2 in CONSISTENCY_CONTRADICTIONS:
        if re.search(p1, lower) and re.search(p2, lower):
            consistency_violations.append(f"{p1} AND {p2}")

    return {
        "has_marker": has_marker,
        "opens_with_slot": opens_with_slot,
        "banned_adjectives_found": banned_found,
        "positional_errors": positional_errors,
        "consistency_violations": consistency_violations,
    }

def print_scorecard(results, label):
    print(f"\n{'='*100}")
    print(f"SCORECARD — {label}")
    print(f"{'='*100}")
    print(f"{'image':<45} {'marker':<8} {'slot':<6} {'banned':<25} {'pos-err':<15} {'consist'}")
    print("-" * 100)
    for img_name, out in results.items():
        s = score(out, img_name)
        banned = ",".join(s["banned_adjectives_found"]) or "none"
        pos = ",".join(s["positional_errors"]) or "none"
        consist = "FAIL" if s["consistency_violations"] else "ok"
        print(f"{img_name:<45} {str(s['has_marker']):<8} {str(s['opens_with_slot']):<6} {banned:<25} {pos:<15} {consist}")

    n = len(results)
    marker_rate = sum(1 for o in results.values() if score(o)['has_marker']) / n
    slot_rate = sum(1 for o in results.values() if score(o)['opens_with_slot']) / n
    no_adj_rate = sum(1 for o in results.values() if not score(o)['banned_adjectives_found']) / n
    no_pos_err = sum(1 for img_name, o in results.items() if not score(o, img_name)['positional_errors']) / n
    no_consist_viol = sum(1 for o in results.values() if not score(o)['consistency_violations']) / n
    print(f"\nAggregate ({n} held-outs):")
    print(f"  marker:                 {marker_rate:.0%}")
    print(f"  slot opener:            {slot_rate:.0%}")
    print(f"  no banned adjectives:   {no_adj_rate:.0%}")
    print(f"  no positional errors:   {no_pos_err:.0%}    [NEW]")
    print(f"  no consistency issues:  {no_consist_viol:.0%}    [NEW]")

print_scorecard(before_dpo_results, "BEFORE DPO (Exp 4b SFT only)")
print_scorecard(after_dpo_results, "AFTER DPO (Exp 4c SFT+DPO)")



SCORECARD — BEFORE DPO (Exp 4b SFT only)
image                                         marker   slot   banned                    pos-err         consist
----------------------------------------------------------------------------------------------------
04-dept-yvelines.png                          True     True   none                      none            ok
baseline-1.png                                True     True   none                      none            ok
browser-share-other-filtered.png              True     True   none                      none            ok
browser-share.png                             True     True   none                      none            ok
chpth-1.png                                   True     True   none                      none            ok
income-vs-life-exp.png                        True     True   none                      none            ok
rural-vs-urban.png                            True     True   none                      none           

In [19]:
# Save full results JSON to Modal Volume (mirrors Exp 4b cell 16)
artifact = {
    "experiment": "exp4c-dpo",
    "date": "2026-04-29",
    "config": {
        "starting_checkpoint": SFT_CHECKPOINT,
        "beta": 0.1,
        "lr": 5e-6,
        "epochs": 3,
        "preference_pair_count": len(pairs),
    },
    "before_dpo": before_dpo_results,
    "after_dpo": after_dpo_results,
}

os.makedirs("/mnt/civicinsight/results", exist_ok=True)
with open("/mnt/civicinsight/results/exp4c-results.json", "w") as f:
    json.dump(artifact, f, indent=2, ensure_ascii=False)
print("Saved to /mnt/civicinsight/results/exp4c-results.json")


Saved to /mnt/civicinsight/results/exp4c-results.json


In [ ]:
# Push DPO adapter to HF private repo as a new commit

REPO_ID = "shahfazal/civicinsight-gemma4-e4b-it"
REVISION = "v0.3-dpo-exp4c"
DPO_CHECKPOINT_DIR = "/mnt/civicinsight/checkpoints/exp4c-dpo"

create_repo(REPO_ID, private=True, repo_type="model", exist_ok=True)

# find the most recent checkpoint dir (DPO might write multiple checkpoints over epochs)
checkpoint_dirs = sorted(
    [d for d in os.listdir(DPO_CHECKPOINT_DIR) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1]),
)
if not checkpoint_dirs:
    raise RuntimeError(f"No checkpoint dirs found in {DPO_CHECKPOINT_DIR}")

final_checkpoint = os.path.join(DPO_CHECKPOINT_DIR, checkpoint_dirs[-1])
print(f"Pushing checkpoint: {final_checkpoint}")

# fix base_model metadata in README before push (same workaround as Exp 4b)
readme_path = os.path.join(final_checkpoint, "README.md")
if os.path.exists(readme_path):
    with open(readme_path) as f:
        content = f.read()
    content = content.replace(
        "base_model: /mnt/civicinsight/model",
        "base_model: unsloth/gemma-4-E4B-it",
    )
    with open(readme_path, "w") as f:
        f.write(content)

api = HfApi()
api.create_branch(repo_id=REPO_ID, branch=REVISION, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path=final_checkpoint,
    repo_id=REPO_ID,
    repo_type="model",
    ignore_patterns=["optimizer.pt", "scheduler.pt", "rng_state.pth"],
    revision=REVISION,
    commit_message="Exp 4c DPO — preference learning on Exp 4c-SFT base",
)
print(f"\nPushed to https://huggingface.co/{REPO_ID}")
